# Step 27 — Is it the model or the sampler? A corrected tilt. A second model family.

**Inputs.** Mount the **Step 19** output. GPU and Internet on. The sections can be switched with the flags below; with all on, about 8–9.5 hours.

**A. Sampler convergence (reviewer concern 2).** The same pretrained model, sampled with Euler–Maruyama at 256, 512 and 1,024 steps in **full precision**. Class frequencies (our classifier and ViT) and FID, against the 256-step half-precision runs already reported. If the imbalance shrinks towards the Heun value as the SDE converges, part of the "model's" mass error belongs to the sampler.

**B. The corrected tilt (concern 6).** Tilting the *model* requires the model's posterior, $P_m(c\mid x)\propto P_d(c\mid x)\,f^m_c/f^d_c$, which gives the log-weight $\mathrm{LSE}(\ell+\log f^t/f^d)-\mathrm{LSE}(\ell+\log f^m/f^d)$. The tilt used previously matched this only to first order. One corrected round, then a second round with the target adjusted by what the first round missed. Optionally (`RUN_BANDS`), the corrected tilt is restricted to $\sigma\ge1$ or $\sigma<1$, the split where the exact test bed gives roughly 0.8/0.4 shares.

**C. A second model family (concern 9).** Google's DDPM CIFAR-10 checkpoint (`google/ddpm-cifar10-32`, Ho et al. 2020), wrapped as an EDM denoiser: $\sigma_t=\sqrt{(1-\bar\alpha_t)/\bar\alpha_t}$, nearest training timestep, $D(x,\sigma)=x-\sigma_t\,\varepsilon_\theta(x\sqrt{\bar\alpha_t},t)$. Sampled with the same reverse SDE and recorded on the same grid. For it we report frequencies, FID, raw and rebalanced agreement with the data, and the **direct reweighting test**: the data, reweighted to DDPM's class frequencies, against DDPM raw.

In [ ]:
RUN_A, RUN_B, RUN_BANDS, RUN_C = True, True, False, True
import os; os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import sys, subprocess, pickle, glob, math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda'; torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 120
N_COND = 2500            # trajectories per tilt condition
CHUNK = 500
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
P19 = glob.glob('/kaggle/input/**/step19_logits.npz', recursive=True); assert P19, 'mount the Step 19 output'
Z19 = np.load(P19[0]); L_TE, W_TE, L_M0, W_M0, SIG_REC = Z19['L_TE'], Z19['W_TE'], Z19['L_M'], Z19['W_M'], Z19['SIG_REC']
SIGMA_MAX, SIGMA_MIN, RHO, STEPS = 80.0, 0.05, 7.0, 256
def edm_sched(steps=STEPS, s_max=SIGMA_MAX, s_min=SIGMA_MIN, rho=RHO):
    i = np.arange(steps); return (s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho) - s_max**(1/rho)))**rho
SCHED = edm_sched(); REC = {int(np.argmin(np.abs(SCHED - s))): gi for gi, s in enumerate(SIG_REC)}
assert np.allclose(SCHED[sorted(REC)], np.sort(SIG_REC)[::-1]), 'recording grid does not match Step 19'
print('reference', L_TE.shape, 'Step 19 baseline trajectories', L_M0.shape)

# ---- classifier used for the reference (from Step 19) ----
class FiLMBlock(nn.Module):
    def __init__(self, cin, cout, emb, stride=1):
        super().__init__(); self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout); self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(cout)
        self.film = nn.Linear(emb, 2*cout); self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout)) if (stride != 1 or cin != cout) else nn.Identity()
    def forward(self, x, e):
        g, b = self.film(e).chunk(2, 1); h = F.silu(self.b1(self.c1(x))); h = self.b2(self.c2(h))*(1+g[:, :, None, None]) + b[:, :, None, None]; return F.silu(h + self.sc(x))
class NoisyResNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(32, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(3, ch, 3, 1, 1)
        self.blocks = nn.ModuleList([FiLMBlock(ch, ch, emb), FiLMBlock(ch, 2*ch, emb, 2), FiLMBlock(2*ch, 2*ch, emb), FiLMBlock(2*ch, 4*ch, emb, 2), FiLMBlock(4*ch, 4*ch, emb)]); self.fc = nn.Linear(4*ch, 10)
    def forward(self, x, sigma):
        f = torch.exp(torch.arange(16, device=x.device)*-(math.log(100)/15)); t = 0.25*torch.log(sigma)[:, None]*f; e = self.emb(torch.cat([torch.sin(t), torch.cos(t)], 1)); h = self.inp(x)
        for b in self.blocks: h = b(h, e)
        return self.fc(h.mean((2, 3)))
CLF = NoisyResNet().to(dev); CLF.load_state_dict(torch.load(glob.glob('/kaggle/input/**/noisy_resnet_seed0.pt', recursive=True)[0], map_location=dev)); CLF.eval()
for p in CLF.parameters(): p.requires_grad_(False)
@torch.no_grad()
def clf_logits(x, sigma, bs=1000):
    out = []
    for xb in x.split(bs):
        with torch.autocast('cuda', dtype=torch.float16): out.append(CLF(xb, torch.full((len(xb),), float(sigma), device=dev)).float())
    return torch.cat(out)

# ---- pretrained EDM ----
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
with dnnlib.util.open_url('https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl') as f: NET = pickle.load(f)['ema'].to(dev).eval()
NET.use_fp16 = True
ACTIVE = {'net': None, 'labels': None}
@torch.no_grad()
def D(x, s):
    if ACTIVE['net'] is None: return NET(x, torch.full((len(x),), float(s), device=dev)).float()
    return ACTIVE['net'](x, torch.full((len(x),), float(s), device=dev), ACTIVE['labels'][:len(x)]).float()
# reference class frequencies (the classifier's labels on test data) = mass-correction target
F_REF = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE); F_BASE19 = np.bincount(W_M0.argmax(1), minlength=10)/len(W_M0)
print('classifier label frequency on test:', np.round(F_REF, 3)); print('Step 19 generated frequency:     ', np.round(F_BASE19, 3))

In [ ]:
def guide_grad(x, s, logr):
    """Exact class-reweighting tilt: grad_x [ logsumexp(l + log r) - logsumexp(l) ] with the noise-conditional classifier."""
    with torch.enable_grad():
        xg = x.detach().requires_grad_(True); l = CLF(xg, torch.full((len(xg),), float(s), device=dev)).float()
        obj = (torch.logsumexp(l + logr, 1) - torch.logsumexp(l, 1)).sum(); return torch.autograd.grad(obj, xg)[0]
@torch.no_grad()
def run_sde(n, seed, logr=None, window=None, lam=1.0, keep_states_at=None):
    """Reverse SDE (Euler-Maruyama, 256 steps) as in Step 19. Returns logits at the recorded levels, terminal weights, final denoised images
    (uint8), and optionally the states at given step indices (for branching)."""
    g = torch.Generator(device=dev).manual_seed(seed); x = torch.randn(n, 3, 32, 32, generator=g, device=dev)*SCHED[0]
    Lrec = np.zeros((len(SIG_REC), n, 10), np.float16); states = {}
    lr = None if logr is None else torch.as_tensor(logr, dtype=torch.float32, device=dev)
    for i in range(STEPS):
        s0 = float(SCHED[i]); s1 = float(SCHED[i+1]) if i+1 < STEPS else SIGMA_MIN
        if i in REC: Lrec[REC[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
        if keep_states_at is not None and i in keep_states_at: states[i] = x.clone()
        if s1 >= s0: continue
        d = D(x, s0); sc = lam*(d - x)/s0**2
        if lr is not None and (window is None or window[0] <= s0 <= window[1]): sc = sc + guide_grad(x, s0, lr)
        x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
    W = F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy(); x0 = D(x, SIGMA_MIN).clamp(-1, 1)
    img = ((x0 + 1)*127.5).round().to(torch.uint8).cpu().numpy()
    return Lrec, W, img, states
def run_condition(name, n, seed0, **kw):
    t0 = time.time(); Ls, Ws, Is = [], [], []
    for c in range(0, n, CHUNK):
        L_, W_, I_, _ = run_sde(min(CHUNK, n - c), seed0 + c, **kw); Ls.append(L_); Ws.append(W_); Is.append(I_)
        if c == 0: print(f"  {name}: first chunk {time.time()-t0:.0f}s -> estimated {(time.time()-t0)*math.ceil(n/CHUNK)/60:.0f} min")
    L_, W_, I_ = np.concatenate(Ls, 1), np.concatenate(Ws), np.concatenate(Is)
    np.savez_compressed(f'/kaggle/working/step27_{name}.npz', L=L_, W=W_, IMG=I_); print(f"  {name}: {len(W_)} trajectories ({(time.time()-t0)/60:.0f} min)"); return L_, W_, I_

def to_float(img): return torch.from_numpy(img).float().div(127.5).sub(1).to(dev)
@torch.no_grad()
def our_labels(img): return np.concatenate([clf_logits(to_float(img[i:i+1000]) + SIGMA_MIN*torch.randn(len(img[i:i+1000]), 3, 32, 32, device=dev), SIGMA_MIN).argmax(1).cpu().numpy() for i in range(0, len(img), 1000)])
# independent strong classifier: ViT fine-tuned on CIFAR-10 (HuggingFace)
VIT = None
try:
    from transformers import AutoModelForImageClassification
    VIT = AutoModelForImageClassification.from_pretrained('aaraki/vit-base-patch16-224-in21k-finetuned-cifar10').to(dev).eval().half()
    names = [VIT.config.id2label[i].lower() for i in range(10)]; VMAP = np.array([CLASSES.index(n) if n in CLASSES else i for i, n in enumerate(names)]); print('ViT labels:', names)
except Exception as e: print('ViT unavailable:', e)
@torch.no_grad()
def vit_labels(img, bs=128):
    out = []
    for i in range(0, len(img), bs):
        x = to_float(img[i:i+bs]); x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)          # [-1,1] == (x-0.5)/0.5 normalisation
        out.append(VIT(pixel_values=x.half()).logits.argmax(1).cpu().numpy())
    return VMAP[np.concatenate(out)]
dst = datasets.CIFAR10('/kaggle/working/cifar', train=False, download=True); TEST_IMG = np.asarray(dst.data).transpose(0, 3, 1, 2); YTE = np.asarray(dst.targets)
if VIT is not None: print(f"ViT test accuracy (2000 images): {np.mean(vit_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
print(f"our classifier test accuracy (sigma_min): {np.mean(our_labels(TEST_IMG[:2000]) == YTE[:2000]):.3f}")
# FID (pytorch-fid Inception) against the 50k training images

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-fid'], check=True)
from pytorch_fid.inception import InceptionV3; from pytorch_fid.fid_score import calculate_frechet_distance
INC = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(dev).eval()
@torch.no_grad()
def inc_stats(img, bs=250):
    f = np.concatenate([INC(torch.from_numpy(img[i:i+bs]).float().div(255).to(dev))[0].squeeze(-1).squeeze(-1).cpu().numpy() for i in range(0, len(img), bs)]); return f.mean(0), np.cov(f, rowvar=False)
dtr = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True); MU_T, S_T = inc_stats(np.asarray(dtr.data).transpose(0, 3, 1, 2))
fid = lambda img: calculate_frechet_distance(*inc_stats(img), MU_T, S_T)

def freq(lbl): return np.bincount(lbl, minlength=10)/len(lbl)
F_VIT_TEST = freq(vit_labels(TEST_IMG)); print('ViT on test set:', np.round(F_VIT_TEST, 3))
def describe(name, W, IMG):
    fo = freq(W.argmax(1)); fv = freq(vit_labels(IMG))
    out = dict(err_our=np.mean(np.abs(np.log(fo/F_REF))), err_vit=np.mean(np.abs(np.log(fv/F_VIT_TEST))), fid=fid(IMG), fv=fv)
    print(f"{name:>30}: class-mass error ours {out['err_our']:.3f}, ViT {out['err_vit']:.3f}; FID (n={len(IMG)}) {out['fid']:.2f}; ViT freqs " + ' '.join(f'{v:.3f}' for v in fv)); return out

## A. Sampler convergence in full precision

In [ ]:
@torch.no_grad()
def run_sched(n, seed, steps, chunk=500):
    """Reverse SDE (EM) with an arbitrary number of steps on the EDM schedule; records logits at the steps nearest the Step 19 grid."""
    sch = edm_sched(steps); rec = {}
    for gi, s in enumerate(SIG_REC): rec.setdefault(int(np.argmin(np.abs(sch - s))), gi)
    Ls, Ws, Is = [], [], []
    for c0 in range(0, n, chunk):
        m = min(chunk, n - c0); g = torch.Generator(device=dev).manual_seed(seed + c0); x = torch.randn(m, 3, 32, 32, generator=g, device=dev)*sch[0]
        Lr = np.zeros((len(SIG_REC), m, 10), np.float16)
        for i in range(steps):
            s0 = float(sch[i]); s1 = float(sch[i+1]) if i+1 < steps else SIGMA_MIN
            if i in rec: Lr[rec[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
            if s1 >= s0: continue
            sc = (D(x, s0) - x)/s0**2; x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
        Ws.append(F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy()); Is.append(((D(x, SIGMA_MIN).clamp(-1, 1) + 1)*127.5).round().to(torch.uint8).cpu().numpy()); Ls.append(Lr)
    return np.concatenate(Ls, 1), np.concatenate(Ws), np.concatenate(Is)
SAMP = {}
if RUN_A:
    NET.use_fp16 = False
    for steps, n in [(256, 2000), (512, 1500), (1024, 1000)]:
        t0 = time.time(); SAMP[steps] = run_sched(n, 90000 + steps, steps); print(f"EM-{steps} fp32: {n} samples ({(time.time()-t0)/60:.0f} min)")
        np.savez_compressed(f'/kaggle/working/step27_em{steps}_fp32.npz', L=SAMP[steps][0], W=SAMP[steps][1], IMG=SAMP[steps][2])
    NET.use_fp16 = True
    print("\nreference points from earlier runs: EM-256 fp16 (Step 21 replicate) ViT error about 0.12-0.15; Heun-18 fp32 about 0.06-0.07")
    for steps in SAMP: describe(f'EM-{steps} fp32', SAMP[steps][1], SAMP[steps][2])

## B. The corrected, iterated tilt

In [ ]:
def corrected_grad(x, s, log_t_over_d, log_m_over_d):
    """grad_x [ LSE(l + log f_t/f_d) - LSE(l + log f_m/f_d) ]: the exact class tilt of the MODEL towards f_t, using a classifier trained on data with prior f_d"""
    with torch.enable_grad():
        xg = x.detach().requires_grad_(True); l = CLF(xg, torch.full((len(xg),), float(s), device=dev)).float()
        obj = (torch.logsumexp(l + log_t_over_d, 1) - torch.logsumexp(l + log_m_over_d, 1)).sum(); return torch.autograd.grad(obj, xg)[0]
@torch.no_grad()
def run_corrected(n, seed, f_t, f_m, band=None, s_split=1.0):
    lt = torch.as_tensor(np.log(f_t/F_REF), dtype=torch.float32, device=dev); lm = torch.as_tensor(np.log(f_m/F_REF), dtype=torch.float32, device=dev)
    Ls, Ws, Is = [], [], []
    for c0 in range(0, n, CHUNK):
        m = min(CHUNK, n - c0); g = torch.Generator(device=dev).manual_seed(seed + c0); x = torch.randn(m, 3, 32, 32, generator=g, device=dev)*SCHED[0]
        Lr = np.zeros((len(SIG_REC), m, 10), np.float16)
        for i in range(STEPS):
            s0 = float(SCHED[i]); s1 = float(SCHED[i+1]) if i+1 < STEPS else SIGMA_MIN
            if i in REC: Lr[REC[i]] = clf_logits(x, s0).cpu().numpy().astype(np.float16)
            if s1 >= s0: continue
            sc = (D(x, s0) - x)/s0**2
            if band is None or (band == 'high' and s0 >= s_split) or (band == 'low' and s0 < s_split): sc = sc + corrected_grad(x, s0, lt, lm)
            x = x + (s1 - s0)*(-2*s0*sc) + math.sqrt(2*s0*(s0 - s1))*torch.randn(x.shape, generator=g, device=dev)
        Ws.append(F.softmax(clf_logits(x, SIGMA_MIN), 1).cpu().numpy()); Is.append(((D(x, SIGMA_MIN).clamp(-1, 1) + 1)*127.5).round().to(torch.uint8).cpu().numpy()); Ls.append(Lr)
    return np.concatenate(Ls, 1), np.concatenate(Ws), np.concatenate(Is)
TILT = {}
if RUN_B:
    t0 = time.time(); TILT['corrected_r1'] = run_corrected(N_COND, 95000, F_REF, F_BASE19); print(f"corrected round 1 ({(time.time()-t0)/60:.0f} min)")
    f1 = freq(TILT['corrected_r1'][1].argmax(1)); f_eff = F_REF*(F_REF/f1); f_eff /= f_eff.sum()
    t0 = time.time(); TILT['corrected_r2'] = run_corrected(N_COND, 96000, f_eff, F_BASE19); print(f"corrected round 2 ({(time.time()-t0)/60:.0f} min)")
    if RUN_BANDS:
        for band in ['high', 'low']:
            for sd in [97000, 98000]: TILT[f'corrected_{band}_s1_{sd}'] = run_corrected(N_COND, sd, F_REF, F_BASE19, band=band, s_split=1.0)
    for k, v in TILT.items(): np.savez_compressed(f'/kaggle/working/step27_{k}.npz', L=v[0], W=v[1], IMG=v[2])
    print("(Step 21 first-order tilt for comparison: class-mass error 0.145 -> 0.082 with our classifier, n=3500)")
    for k, v in TILT.items(): describe(k, v[1], v[2])

## C. DDPM (Ho et al. 2020) as a second model family

In [ ]:
DDPM = {}
if RUN_C:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers'], check=True)
    from diffusers import UNet2DModel
    UNET = UNet2DModel.from_pretrained('google/ddpm-cifar10-32').to(dev).eval()
    betas = torch.linspace(1e-4, 0.02, 1000, dtype=torch.float64); abar = torch.cumprod(1 - betas, 0).numpy(); SIG_T = np.sqrt((1 - abar)/abar)
    @torch.no_grad()
    def D_ddpm(x, s):
        t = int(np.argmin(np.abs(np.log(SIG_T) - np.log(s)))); st = float(SIG_T[t]); a = float(abar[t])
        with torch.autocast('cuda', dtype=torch.float16): eps = UNET(x*math.sqrt(a), torch.full((len(x),), t, device=dev)).sample.float()
        return x - st*eps
    print(f"DDPM noise range: sigma_t from {SIG_T[0]:.4f} to {SIG_T[-1]:.1f}")
    D_saved = D; D = D_ddpm; t0 = time.time()
    DDPM['L'], DDPM['W'], DDPM['IMG'] = run_sched(3000, 99000, 256)
    D = D_saved; print(f"DDPM: 3000 trajectories ({(time.time()-t0)/60:.0f} min)")
    np.savez_compressed('/kaggle/working/step27_ddpm.npz', **DDPM)
    describe('DDPM, EM-256', DDPM['W'], DDPM['IMG'])

## Evaluation against the data (same pipeline as before), including the direct reweighting test

In [ ]:
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_h(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG_REC))] for a, b in PAIRS}
def pair_sh(L, W, a, b, idx, balance=False):
    """(sigma_bif, h) along the pair log-odds, standardised by the reference, weighted by terminal class weights."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if balance: tot = 0.5*(Wa.sum() + Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    prev = None
    for g in range(len(SIG_REC)):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: prev = 1; continue
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; sm, p = wkde_score((z - mid)/half, Wa + Wb); nm = n_max(stationary(sm, p))
        if nm >= 2 and prev is not None: return float(SIG_REC[g]), cusp_h(sm, p)
        if nm == 1: prev = 1
    return None, None
def evaluate(L, W, name, NB=40):
    n = len(W); ir = rng.choice(len(W_TE), size=n, replace=False); allm = np.arange(n); rows = []
    fm = np.bincount(W.argmax(1), minlength=10)/n; over = np.log(fm/F_REF)
    for a, b in PAIRS:
        sm_, hm = pair_sh(L, W, a, b, allm); sr, hr = pair_sh(L_TE, W_TE, a, b, ir); _, hmb = pair_sh(L, W, a, b, allm, balance=True)
        if sm_ is None or sr is None: continue
        rat = []
        for _ in range(NB):
            s1, _ = pair_sh(L, W, a, b, rng.integers(n, size=n)); s2, _ = pair_sh(L_TE, W_TE, a, b, rng.choice(len(W_TE), size=n))
            if s1 and s2: rat.append(s1/s2)
        lo, hi = np.percentile(rat, [2.5, 97.5]) if len(rat) > 10 else (np.nan, np.nan)
        rows.append(dict(pair=(a, b), sm=sm_, sr=sr, hm=hm, hr=hr, hmb=hmb, lo=lo, hi=hi, dover=over[a]-over[b]))
    R = np.array([r['sm']/r['sr'] for r in rows]); hm = np.array([r['hm'] for r in rows]); hr = np.array([r['hr'] for r in rows]); hmb = np.array([r['hmb'] for r in rows]); dov = np.array([r['dover'] for r in rows])
    lo = np.array([r['lo'] for r in rows]); hi = np.array([r['hi'] for r in rows])
    print(f"{name:>20}: pairs {len(rows)}; sigma_bif ratio median {np.median(R):.2f} (CI>1: {(lo>1).sum()}, CI<1: {(hi<1).sum()}); corr h model-data raw {np.corrcoef(hm, hr)[0,1]:.2f}, rebalanced {np.corrcoef(hmb, hr)[0,1]:.2f}; "
          f"corr h model vs over-generation {np.corrcoef(hm, dov)[0,1]:.2f}; mean |log over-generation| {np.mean(np.abs(over)):.3f}")
    return {r['pair']: r for r in rows}, fm

SIG = SIG_REC
L_TR, W_TR = Z19['L_TR'], Z19['W_TR']
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3, c1 + 3*c3*s**2 + 2*c2*s, b
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG))] for a, b in PAIRS}
def analyse(L, W, a, b, idx, balance=False, rw=None, g0=0, slope=False):
    """Along the pair log-odds (reference standardisation): returns dict(sigma, gi, h, a, b[, s]). rw: extra class reweighting factors (for part C)."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if rw is not None: Wa = Wa*rw[a]; Wb = Wb*rw[b]
    if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    def dist(g):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: return None
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; return wkde_score((z - mid)/half, Wa + Wb)
    prev = None if g0 == 0 else 1
    for g in range(g0, len(SIG)):
        d = dist(g); nm = 1 if d is None else n_max(stationary(*d))
        if nm >= 2 and prev is not None:
            h, a_, b_ = cusp(*d); out = dict(sigma=float(SIG[g]), gi=g, h=h, a=a_, b=b_)
            if slope:                                            # a(log sigma) near the transition, from the levels just above and below
                gs = [k for k in range(max(0, g-4), min(len(SIG), g+3)) if dist(k) is not None]
                av = [cusp(*dist(k))[1] for k in gs]
                if len(gs) >= 3: out['s'] = -np.polyfit(np.log(SIG[gs]), av, 1)[0]
            return out
        if nm == 1: prev = 1
    return None
def boot(L, W, a, b, n, B, balance=False, ref=False, g_hint=None):
    """Bootstrap medians and 95% intervals of sigma_bif and h at sample size n."""
    ss, hs = [], []; g0 = 0 if g_hint is None else max(0, g_hint - 8)
    for _ in range(B):
        idx = rng.choice(len(W), size=n, replace=True); r = analyse(L, W, a, b, idx, balance, g0=g0)
        if r: ss.append(r['sigma']); hs.append(r['h'])
    if len(ss) < B//3: return None
    return dict(sigma=np.median(ss), s_lo=np.percentile(ss, 2.5), s_hi=np.percentile(ss, 97.5), h=np.median(hs), h_lo=np.percentile(hs, 2.5), h_hi=np.percentile(hs, 97.5), n_ok=len(ss))
def class_block_ci(x, y, pairs, B=5000, fn=None):
    """Class-block bootstrap: resample the 10 classes with replacement, weight each pair by the product of its classes' multiplicities."""
    P = np.array(pairs); out = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() < 6: continue
        if fn is None:
            mx, my = np.average(x, weights=w), np.average(y, weights=w); out.append(np.sum(w*(x-mx)*(y-my))/np.sqrt(np.sum(w*(x-mx)**2)*np.sum(w*(y-my)**2)))
        else: out.append(fn(w))
    out = np.array(out, float); out = out[np.isfinite(out)]
    return np.percentile(out, [2.5, 97.5]) if len(out) > 20 else np.array([np.nan, np.nan])

def bmed(L, W, a, b, n, B=100, balance=False, rw=None):
    r0 = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=len(W) < n), balance, rw)
    if r0 is None: return None
    out = []
    for _ in range(B):
        r = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=True), balance, rw, g0=max(0, r0['gi'] - 8))
        if r: out.append((r['sigma'], r['h']))
    if len(out) < B//3: return None
    o = np.array(out); return np.median(o[:, 0]), np.median(o[:, 1])
def full_eval(name, L, W, B=100):
    n = len(W); fm = np.bincount(W.argmax(1), minlength=10)/n; RW = fm/F_REF; rows = {}
    for p in PAIRS:
        m = bmed(L, W, *p, n, B); mb = bmed(L, W, *p, n, B, balance=True); d = bmed(L_TE, W_TE, *p, n, B); drw = bmed(L_TE, W_TE, *p, n, B, rw=RW)
        if m and mb and d and drw: rows[p] = (m, mb, d, drw)
    P_ = list(rows); g = lambda k, j: np.array([rows[p][k][j] for p in P_])
    lr = lambda k1, k2: np.log(g(k1, 0)/g(k2, 0))
    print(f"\n[{name}] {len(P_)} pairs, n={n}")
    for lab, k1, k2 in [('model raw vs data', 0, 2), ('model rebalanced vs data', 1, 2), ('model raw vs data reweighted to model mass', 0, 3)]:
        c = class_block_ci(g(k1, 1), g(k2, 1), P_, B=1000); s = class_block_ci(None, None, P_, B=1000, fn=lambda w, x=lr(k1, k2): np.sqrt(np.average((x - np.average(x, weights=w))**2, weights=w)))
        print(f"   {lab:>44}: corr h {np.corrcoef(g(k1, 1), g(k2, 1))[0,1]:.3f} [{c[0]:.2f}, {c[1]:.2f}]; location SD {lr(k1, k2).std():.3f} [{s[0]:.3f}, {s[1]:.3f}]; geometric-mean ratio {np.exp(lr(k1, k2).mean()):.3f}")
    return rows
EV = {}
if RUN_C and DDPM: EV['DDPM'] = full_eval('DDPM (Ho et al.), EM-256', DDPM['L'], DDPM['W'])
if RUN_B:
    for k in ['corrected_r1', 'corrected_r2']: EV[k] = full_eval(k, TILT[k][0], TILT[k][1])
if RUN_A and 256 in SAMP: EV['em256_fp32'] = full_eval('EDM, EM-256 fp32', SAMP[256][0], SAMP[256][1])
# floor at a matching n: train vs test reference
nfl = 3000; fl = {p: (bmed(L_TR, W_TR, *p, nfl), bmed(L_TE, W_TE, *p, nfl)) for p in PAIRS}; fl = {p: v for p, v in fl.items() if v[0] and v[1]}
P_ = list(fl); x = np.log(np.array([fl[p][0][0]/fl[p][1][0] for p in P_])); hc = np.corrcoef([fl[p][0][1] for p in P_], [fl[p][1][1] for p in P_])[0, 1]
print(f"\nfloor at n={nfl} (train vs test reference): location SD {x.std():.3f}; corr h {hc:.3f}")

## What to report back
- **A:** the three `EM-… fp32` lines.
- **B:** the corrected round lines, plus the band lines if `RUN_BANDS` was on.
- **C:** the DDPM line.
- **Evaluation:** each evaluation block, and the floor line.